# Beni wake-word training (Colab)

End-to-end notebook that trains a microWakeWord model for the phrase
**"Hey Beni"** and exports a streaming int8 TFLite the ESP32-S3
firmware can load directly.

This notebook is a **thin adaptation** of the upstream reference at
`kahrendt/microWakeWord/notebooks/basic_training_notebook.ipynb`.
If you need to tune something and this notebook doesn't cover it,
diff against upstream — we intentionally stay close to it so
cross-referencing the docs works.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (or A100 if available).
2. Run all cells top-to-bottom. After the install cell you'll need
   to **Restart Session** — Colab will prompt.
3. The final cell downloads `stream_state_internal_quant.tflite`
   and a ready-to-include `hey_beni_model.h`. Drop both into
   `firmware/main/wake_word/model/`.

Total runtime: roughly 1-2 h on a T4 — most of it spent on the
background-noise downloads and augmentation feature extraction, not
training itself.

## 1. Install microWakeWord

The package is not on PyPI — it's distributed from the author's
GitHub. We also pull a patched `audio-metadata` (upstream broke on
recent `attrs` versions, the fork un-pins it) to avoid a cryptic
import error during data loading.

**After this cell finishes, Colab will ask to restart the session.
Do it.** TensorFlow is reloaded from microwakeword's pinned version
and the current kernel has stale C-extensions.

In [ ]:
import platform

if platform.system() == 'Darwin':
    # Not our target env (Colab = Linux) but handy if a dev runs this
    # locally on a Mac for smoke testing.
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

# `attrs` fork keeps audio-metadata working across TF's pinned attrs.
!pip install -q 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

!git clone https://github.com/kahrendt/microWakeWord
!pip install -q -e ./microWakeWord

# Sanity check.
!nvidia-smi -L || echo '(no GPU — training will crawl on CPU)'

## 2. Generate 1 sample of "Hey Beni" to verify Piper settings

piper-sample-generator takes an English phrase and renders many
voiced variants by sampling from the LibriTTS neural voice. "beni"
reads phonetically correctly in English ("BEN-ee") so we don't need
a Russian voice for v1 — the wake detector is acoustic, not
linguistic. If later we want a Russian voice, swap the model file
below for a Piper ru voice.

Listen to the output of this cell to sanity-check the pronunciation
**before** generating thousands of samples.

In [ ]:
import os
from IPython.display import Audio

TARGET_WORD = 'hey beni'  # Phonetic; matches how callers will say it

# piper-sample-generator is on PyPI, but v3.2.0's wheel is broken:
# `pyproject.toml` restricts packaging to `piper_sample_generator*`,
# which excludes the sibling `piper_train` package — yet
# `piper_sample_generator.__main__` imports from it (`from
# piper_train.vits import commons`). So `pip install` alone gives
# you `ModuleNotFoundError: piper_train` at first invocation.
#
# Workaround: install the wheel for its runtime deps, then clone
# the source repo and copy `piper_train/` into cwd. At
# `python3 -m piper_sample_generator` time, cwd is the first entry
# on sys.path, so the top-level `piper_train` package resolves.
!pip install -q piper-sample-generator

if not os.path.exists('piper_train'):
    !git clone --depth 1 https://github.com/rhasspy/piper-sample-generator /tmp/psg_src
    !cp -r /tmp/psg_src/piper_train .

MODEL_PATH = 'piper_models/en_US-libritts_r-medium.pt'
if not os.path.exists(MODEL_PATH):
    os.makedirs('piper_models', exist_ok=True)
    !wget -q -O {MODEL_PATH} \
        'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

!python3 -m piper_sample_generator "{TARGET_WORD}" \
    --model {MODEL_PATH} \
    --max-samples 1 --batch-size 1 --output-dir generated_samples

Audio('generated_samples/0.wav', autoplay=True)

## 3. Generate ~2000 wake-word samples

Piper's default parameters produce a reasonable prosody spread —
different speakers, rates, pitches. If after training the model is
too narrow (works on some voices, not others), crank `--max-samples`
and replay. Also worth experimenting with `--noise-scales` for
broader prosody jitter; see piper-sample-generator's README.

`2000` was chosen as a sensible default for a 1-2 word wake phrase.
Upstream notebook's default is `1000` but elderly-care is a harder
problem (slurred speech, background TV) so we over-sample.

In [ ]:
!python3 -m piper_sample_generator "{TARGET_WORD}" \
    --model {MODEL_PATH} \
    --max-samples 2000 --batch-size 100 --output-dir generated_samples

import glob
print('positive samples:', len(glob.glob('generated_samples/*.wav')))

## 4. Download augmentation data (RIRs, background noise)

Three sources, all routed through HuggingFace's `datasets`:

* **MIT impulse responses** — convolved with positives to simulate
  room reverb. Makes the model robust to hearing "Hey Beni" from
  across a kitchen vs. directly into the mic.
* **AudioSet (single shard)** — general environmental noise. One
  shard is enough for v1; crank more if FRR under noise is bad.
* **Free Music Archive (tiny subset)** — catches the case where the
  TV/radio is on during a call.

**License note:** AudioSet / FMA are research-only; if we ship
commercially we either swap these for licensed alternatives or rely
on augmentation-free robustness (much weaker). For prototyping,
this matches upstream and is fine.

In [ ]:
import datasets, scipy, os, numpy as np
from pathlib import Path
from tqdm import tqdm

# --- MIT RIRs ---
output_dir = './mit_rirs'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    rir_dataset = datasets.load_dataset(
        'davidscripka/MIT_environmental_impulse_responses', split='train', streaming=True)
    for row in tqdm(rir_dataset, desc='RIRs'):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(
            os.path.join(output_dir, name), 16000,
            (row['audio']['array']*32767).astype(np.int16))

# --- AudioSet (one shard ~ 1 GB) ---
if not os.path.exists('audioset'):
    os.mkdir('audioset')
    fname = 'bal_train09.tar'
    link = 'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/' + fname
    !wget -q -O audioset/{fname} {link}
    !cd audioset && tar -xf {fname}
    os.makedirs('audioset_16k', exist_ok=True)
    audioset_ds = datasets.Dataset.from_dict({'audio': [str(i) for i in Path('audioset/audio').glob('**/*.flac')]})
    audioset_ds = audioset_ds.cast_column('audio', datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_ds, desc='AudioSet→16k'):
        name = row['audio']['path'].split('/')[-1].replace('.flac', '.wav')
        scipy.io.wavfile.write(
            os.path.join('audioset_16k', name), 16000,
            (row['audio']['array']*32767).astype(np.int16))

# --- FMA tiny subset (music backdrop) ---
if not os.path.exists('fma_16k'):
    os.mkdir('fma_16k')
    # mchl914's xs subset ~ 7 GB full; we only need a slice for v1.
    fma_ds = datasets.load_dataset('mchl914/free_music_archive_xs', split='train', streaming=True)
    count = 0
    for row in tqdm(fma_ds, desc='FMA→16k'):
        if count >= 400:  # Plenty for background augmentation.
            break
        arr = row['audio']['array']
        if row['audio']['sampling_rate'] != 16000:
            # `datasets` already resamples on cast; defensive only.
            continue
        scipy.io.wavfile.write(
            os.path.join('fma_16k', f'{count}.wav'), 16000,
            (arr*32767).astype(np.int16))
        count += 1

## 5. Wire up Clips + Augmentation

The `Clips` class loads our Piper samples with a 10% holdout split;
`Augmentation` defines the probabilistic pipeline. Settings match
upstream defaults — they're already tuned for wake-word detection,
reinventing is how you get bad recall.

In [ ]:
from microwakeword.audio.clips import Clips
from microwakeword.audio.augmentation import Augmentation

clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.1,
        'TanhDistortion': 0.1,
        'PitchShift': 0.1,
        'BandStopFilter': 0.1,
        'AddColorNoise': 0.1,
        'AddBackgroundNoise': 0.75,
        'Gain': 1.0,
        'RIR': 0.5,
    },
    impulse_paths=['mit_rirs'],
    background_paths=['fma_16k', 'audioset_16k'],
    background_min_snr_db=-5,
    background_max_snr_db=10,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

# Quick sanity listen: augment a random clip.
from microwakeword.audio.audio_utils import save_clip
from IPython.display import Audio
random_clip = clips.get_random_clip()
augmented = augmenter.augment_clip(random_clip)
save_clip(augmented, 'augmented_clip.wav')
Audio('augmented_clip.wav', autoplay=True)

## 6. Generate augmented spectrograms (train / val / test)

microWakeWord stores features as memory-mapped ragged arrays so
training doesn't re-synthesize spectrograms each epoch. This cell
is the most time-expensive — ~20 min on a T4 for 2k positives.

In [ ]:
import os
from mmap_ninja.ragged import RaggedMmap
from microwakeword.audio.spectrograms import SpectrogramGeneration

output_dir = 'generated_augmented_features'
os.makedirs(output_dir, exist_ok=True)

for split in ('training', 'validation', 'testing'):
    out_dir = os.path.join(output_dir, split)
    os.makedirs(out_dir, exist_ok=True)

    if split == 'training':
        split_name, repetition, slide = 'train', 2, 10
    elif split == 'validation':
        split_name, repetition, slide = 'validation', 1, 10
    else:  # testing uses streaming-style single-slide
        split_name, repetition, slide = 'test', 1, 1

    spectrograms = SpectrogramGeneration(
        clips=clips,
        augmenter=augmenter,
        slide_frames=slide,
        step_ms=10,
    )

    RaggedMmap.from_generator(
        out_dir=os.path.join(out_dir, 'wakeword_mmap'),
        sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=repetition),
        batch_size=100,
        verbose=True,
    )

## 7. Download pre-generated negative spectrograms

The upstream author (Kevin Ahrendt) publishes pre-computed negative
feature sets — speech, dinner-party noise, no-speech ambient — on
HuggingFace. Using these beats spinning our own because:

1. The spectrogram front-end **must** match byte-for-byte what the
   firmware runs. Using Kevin's sets means we use his extractor by
   construction.
2. They're already sliced into hard-negative categories the trainer
   expects: `dinner_party` stress-tests against cocktail-party
   audio, `speech` against general English, `no_speech` against
   silence/ambient — one training run covers all failure modes.

In [ ]:
output_dir = './negative_datasets'
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
    link_root = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/'
    for fname in ('dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip'):
        !wget -q -O negative_datasets/{fname} {link_root}{fname}
        !unzip -q negative_datasets/{fname} -d {output_dir}

## 8. Write the training YAML

microWakeWord's trainer is a CLI that reads a YAML describing the
feature mix — which sets to sample from, with what weight, and
whether they're truth (positive) or distractor (negative).

The weights below mirror upstream. The asymmetry (10× speech vs. 2×
positives) is intentional: hard negatives are scarce in the real
world but catastrophic when missed — over-sampling teaches the
model to push confidently past them.

In [ ]:
import yaml

config = {
    'window_step_ms': 10,
    'train_dir': 'trained_models/wakeword',
    'features': [
        {
            'features_dir': 'generated_augmented_features',
            'sampling_weight': 2.0, 'penalty_weight': 1.0,
            'truth': True, 'truncation_strategy': 'truncate_start', 'type': 'mmap',
        },
        {
            'features_dir': 'negative_datasets/speech',
            'sampling_weight': 10.0, 'penalty_weight': 1.0,
            'truth': False, 'truncation_strategy': 'random', 'type': 'mmap',
        },
        {
            'features_dir': 'negative_datasets/dinner_party',
            'sampling_weight': 10.0, 'penalty_weight': 1.0,
            'truth': False, 'truncation_strategy': 'random', 'type': 'mmap',
        },
        {
            'features_dir': 'negative_datasets/no_speech',
            'sampling_weight': 5.0, 'penalty_weight': 1.0,
            'truth': False, 'truncation_strategy': 'random', 'type': 'mmap',
        },
        {  # Held-out eval set — weight 0 so it's never sampled for training.
            'features_dir': 'negative_datasets/dinner_party_eval',
            'sampling_weight': 0.0, 'penalty_weight': 1.0,
            'truth': False, 'truncation_strategy': 'split', 'type': 'mmap',
        },
    ],
    # Training loop hyperparameters. These are sensible defaults for
    # wake-word training; change if val loss plateaus unusually high.
    'training_steps': '10000,10000,10000',
    'positive_class_weight': '1,1,1',
    'negative_class_weight': '20,20,20',
    'learning_rates': '0.001,0.0005,0.00025',
    'batch_size': 128,
    'time_mask_max_size': '0,5,10',
    'time_mask_count': '0,1,2',
    'freq_mask_max_size': '0,5,10',
    'freq_mask_count': '0,1,2',
    'eval_step_interval': 500,
    'clip_duration_ms': 1500,
    'target_minimization': 0.9,
    'minimization_metric': None,
    'maximization_metric': 'average_viable_recall',
}

with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f)

print('wrote training_parameters.yaml')

## 9. Train

Launches the MixedNet trainer. Takes 30-60 min on a T4. It prints
nothing for several minutes at the start (warm-up + data indexing)
— this is normal, don't kill it.

At the end you get an int8 streaming TFLite at
`trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite`
plus a printed FRR/FAR table the firmware's threshold will be
picked from.

In [ ]:
!python -m microwakeword.model_train_eval \
    --training_config='training_parameters.yaml' \
    --train 1 \
    --restore_checkpoint 1 \
    --test_tf_nonstreaming 0 \
    --test_tflite_nonstreaming 0 \
    --test_tflite_nonstreaming_quantized 0 \
    --test_tflite_streaming 0 \
    --test_tflite_streaming_quantized 1 \
    --use_weights 'best_weights' \
    mixednet \
    --pointwise_filters '64,64,64,64' \
    --repeat_in_block '1,1,1,1' \
    --mixconv_kernel_sizes '[5],[7,11],[9,15],[23]' \
    --residual_connection '0,0,0,0' \
    --first_conv_filters 32 \
    --first_conv_kernel_size 5 \
    --stride 3

## 10. Emit C header + download artifacts

The trainer writes the `.tflite` to disk. We additionally dump a
C header (compile-time byte array) because esp-tflite-micro on
ESP32-S3 can't mmap models from SPIFFS without extra plumbing, and
embedding the model in flash is simpler for v1.

Then trigger two browser downloads — drop both into
`firmware/main/wake_word/model/`.

In [ ]:
from pathlib import Path

# Inlined from scripts/dump_c_header.py — Colab's "Open from GitHub"
# fetches only the notebook, not sibling files, so we can't `import`
# from the repo. If you edit this, sync the .py version too.
def dump_as_c_header(tflite_path: Path, header_path: Path, var_name: str) -> None:
    raw = Path(tflite_path).read_bytes()
    lines = [
        "// Auto-generated by ml/wake_word/train_beni.ipynb.",
        "// DO NOT EDIT: regenerate by re-running the training notebook.",
        "#pragma once",
        "#include <stddef.h>",
        "#include <stdint.h>",
        "",
        f"extern const unsigned char {var_name}[];",
        f"extern const size_t {var_name}_len;",
        "",
        # aligned(8) is required — esp-tflite-micro reads the
        # flatbuffer directly from flash, and misaligned access
        # faults on ESP32-S3.
        f"const unsigned char {var_name}[] __attribute__((aligned(8))) = {{",
    ]
    for i in range(0, len(raw), 12):
        chunk = raw[i : i + 12]
        lines.append("  " + ", ".join(f"0x{b:02x}" for b in chunk) + ",")
    lines.append("};")
    lines.append(f"const size_t {var_name}_len = {len(raw)};")
    lines.append("")
    Path(header_path).parent.mkdir(parents=True, exist_ok=True)
    Path(header_path).write_text("\n".join(lines))


TFLITE = Path('trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite')
HEADER = Path('hey_beni_model.h')

assert TFLITE.exists(), f'expected trained model at {TFLITE} — did training succeed?'
dump_as_c_header(TFLITE, HEADER, var_name='hey_beni_tflite')

print(f'tflite: {TFLITE.stat().st_size/1024:.1f} kB')
print(f'header: {HEADER.stat().st_size/1024:.1f} kB')

In [ ]:
from google.colab import files
files.download(str(TFLITE))
files.download(str(HEADER))